# Этап 4: Реальное квантование (Weight-only INT8 Storage)

Здесь мы уходим от симуляции и физически меняем тип данных в памяти. Мы будем хранить веса как `int8`, а для вычислений — превращать их обратно в `float16/32` на лету.

In [11]:
# Настройка путей: если мы в подпапке, переходим в корень проекта

import torch
from src.model import GPTLanguageModel, device
from src.utils import encode, decode

# 1. Загружаем оригинал
model = GPTLanguageModel().to(device)
model.load_state_dict(torch.load('nanoGPT-lab/model_ckpt.pt', map_location=device))
model.eval()

# Функции для реального квантования в словарь
def compress_to_int8(state_dict, sensitive_layers=[]):
    compressed_dict = {}
    scales = {}
    
    for name, param in state_dict.items():
        if 'weight' in name and param.dim() > 1:
            if name not in sensitive_layers:
                # Находим масштаб
                x_max = param.abs().max().item()
                scale = x_max / 127.0
                
                # Физически кастуем в int8
                q_param = torch.round(param / scale).to(torch.int8)
                
                compressed_dict[name] = q_param
                scales[name + "_scale"] = torch.tensor(scale)
                print(f"📦 Сжат слой {name}: {param.dtype} -> {q_param.dtype}")
            else:
                 # Чувствительные слои оставляем float32 (или float16, если хотим сэкономить)
                compressed_dict[name] = param
                print(f"⚠️ Пропущен слой (Sensitive): {name} -> {param.dtype}")
        else:
            # Остальное (LayerNorm, Bias) оставляем как есть
            compressed_dict[name] = param
            
    return compressed_dict, scales

sensitive_layers = ['token_embedding_table.weight', 'position_embedding_table.weight', 'lm_head.weight']
compressed_weights, scales = compress_to_int8(model.state_dict(), sensitive_layers=sensitive_layers)

⚠️ Пропущен слой (Sensitive): token_embedding_table.weight -> torch.float32
⚠️ Пропущен слой (Sensitive): position_embedding_table.weight -> torch.float32
📦 Сжат слой blocks.0.sa.heads.0.key.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.0.query.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.0.value.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.1.key.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.1.query.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.1.value.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.2.key.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.2.query.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.2.value.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.3.key.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.3.query.weight: torch.float32 -> torch.int8
📦 Сжат слой blocks.0.sa.heads.3.value.w

### 2. Сравнение размера файлов на диске
Это самый честный способ проверить сжатие.

In [12]:
import os

torch.save(model.state_dict(), 'nanoGPT-lab/model_fp32.pt')
torch.save({'weights': compressed_weights, 'scales': scales}, 'nanoGPT-lab/model_int8_real.pt')

size_fp32 = os.path.getsize('nanoGPT-lab/model_fp32.pt') / 1024**2
size_int8 = os.path.getsize('nanoGPT-lab/model_int8_real.pt') / 1024**2

print(f"Размер файла FP32: {size_fp32:.2f} MB")
print(f"Размер файла INT8: {size_int8:.2f} MB")
print(f"Реальная экономия места: {size_fp32/size_int8:.1f}x")

Размер файла FP32: 50.23 MB
Размер файла INT8: 19.88 MB
Реальная экономия места: 2.5x


### 3. Как запустить такую модель? (Dequantize-on-the-fly)
Мы не можем просто сделать `model.load_state_dict()`, так как типы не совпадут. Нам нужно деквантовать веса перед загрузкой.

In [13]:
def decompress_int8(compressed_data):
    weights = compressed_data['weights']
    scales = compressed_data['scales']
    new_state_dict = {}
    
    for name, param in weights.items():
        if name + "_scale" in scales:
            # Восстанавливаем из int8 во float32
            scale = scales[name + "_scale"]
            new_state_dict[name] = param.to(torch.float32) * scale
        else:
            new_state_dict[name] = param
            
    return new_state_dict

# Эмуляция загрузки сжатой модели
checkpoint = torch.load('model_int8_real.pt', map_location=device)
recovered_state = decompress_int8(checkpoint)

model_int8_recovered = GPTLanguageModel().to(device)
model_int8_recovered.load_state_dict(recovered_state)
model_int8_recovered.eval()

print("✅ Модель успешно восстановлена из INT8 и готова к работе!")

✅ Модель успешно восстановлена из INT8 и готова к работе!


In [14]:
# Настройка путей: если мы в подпапке, переходим в корень проекта

import torch
import time
import pandas as pd
from src.model import GPTLanguageModel, device
from src.utils import encode, decode, estimate_loss

In [15]:

def measure_performance(mdl, num_tokens=50):
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    
    # Warmup
    _ = mdl.generate(context, max_new_tokens=5)
    
    # 1. Latency (TTFT)
    start_latency = time.time()
    _ = mdl.generate(context, max_new_tokens=1)
    latency = (time.time() - start_latency) * 1000
    
    # 2. Throughput
    start_throughput = time.time()
    _ = mdl.generate(context, max_new_tokens=num_tokens)
    duration = time.time() - start_throughput
    throughput = num_tokens / duration
    
    return latency, throughput

print("Evaluating Original FP32 Model...")
loss_fp32 = estimate_loss(model)['val'].item()
lat_fp32, thr_fp32 = measure_performance(model)
# Размер файла мы уже знаем из предыдущих шагов

print("Evaluating Recovered INT8 Model...")
loss_int8 = estimate_loss(model_int8_recovered)['val'].item()
lat_int8, thr_int8 = measure_performance(model_int8_recovered)
# Размер файла INT8 тоже знаем

results = {
    "Metric": ["Disk Size (MB)", "Throughput (tokens/s)", "Latency (ms)", "Validation Loss"],
    "Original FP32": [size_fp32, thr_fp32, lat_fp32, loss_fp32],
    "Recovered INT8": [size_int8, thr_int8, lat_int8, loss_int8],
    "Delta": [
        f"{size_fp32/size_int8:.1f}x smaller", 
        f"{(thr_int8/thr_fp32 - 1)*100:+.1f}%", 
        f"{lat_int8 - lat_fp32:+.2f} ms", 
        f"{(loss_int8/loss_fp32 - 1)*100:+.2f}%"
    ]
}

df = pd.DataFrame(results)
display(df)


Evaluating Original FP32 Model...
Evaluating Recovered INT8 Model...


,Metric,Original FP32,Recovered INT8,Delta
0,Disk Size (MB),50.230075,19.881484,2.5x smaller
1,Throughput (tokens/s),42.896474,61.256434,+42.8%
2,Latency (ms),11.931896,11.931896,+0.00 ms
3,Validation Loss,1.505717,1.501790,-0.26%


### 4. Сравнение генерации (Baseline vs Recovered)
Убедимся, что после физического цикла сжатия-распаковки качество осталось таким же, как при Fake Quantization.

In [10]:
import torch
from src.model import GPTLanguageModel, device
from src.utils import encode, decode
from src.inference import generate_stream

# --- 1. Подгружаем модели с диска ---
# 1.1 Original FP32
model_fp32 = GPTLanguageModel().to(device)
try:
    # Загружаем оригинальный чекпоинт
    model_fp32.load_state_dict(torch.load('nanoGPT-lab/model_ckpt.pt', map_location=device))
    model_fp32.eval()
    print("✅ Loaded Original FP32 Model")
except FileNotFoundError:
    print("⚠️ FP32 checkpoint not found!")
    
# 1.2 Recovered INT8
# Нам нужно сначала загрузить сжатые данные, а потом их распаковать
# (функция decompress_int8 должна быть определена выше в ноутбуке)
model_int8 = GPTLanguageModel().to(device)
try:
    checkpoint_int8 = torch.load('model_int8_real.pt', map_location=device)
    recovered_state_dict = decompress_int8(checkpoint_int8) # Используем функцию из ячейки r6
    model_int8.load_state_dict(recovered_state_dict)
    model_int8.eval()
    print("✅ Loaded Recovered INT8 Model")
except FileNotFoundError:
    print("⚠️ INT8 checkpoint not found (run step 2 first)!")


prompt = "JULIET: "
context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
print("\n--- ORIGINAL FP32 OUTPUT ---")
# Используем потоковую генерацию для эффекта печатающей машинки
print(prompt, end='')

# Используем потоковую генерацию
try:
    for token_id in generate_stream(model_fp32, context, max_new_tokens=1500):
        print(decode([token_id]), end='', flush=True)
except Exception as e:
    print(f"\nError in FP32 generation: {e}")
print("\n")

print("\n--- RECOVERED INT8 OUTPUT ---")
print(prompt, end='')
try:
    for token_id in generate_stream(model_int8, context, max_new_tokens=1500):
        print(decode([token_id]), end='', flush=True)
except Exception as e:
    print(f"\nError in INT8 generation: {e}")
print("\n")

✅ Loaded Original FP32 Model
✅ Loaded Recovered INT8 Model

--- ORIGINAL FP32 OUTPUT ---
JULIET: what's do?
Look you; I'll stay at thee thou betcher with my master.

Nurse:
Sir, he canst have thee and myself wherein my graves?

ROMEO:
On my counsel cousin!

Nurse:
Injured, what time of me?

ROMEO:
I'll beging.

JULIET:
I what, no high boy?

Nurse:
Read is the curse o' the guilty of foright,
Give me thee I this wille.

JULIET:
Having me so told of it.
And did teach oppose of him,
Make make my thronour hath a bosom.

JULIET:
Give me not like my church-sun, and young be hence.

JULIET:
On dear me, and, sweet you dare knee
By the own of daughters, twits but myself.

ROMEO:
Peace, faults, how fearful with her!

Nurse:
What dogs you might, I will be, sir,
Be infond in the little Richard.
Good fellow, boy, farewell;
The gift of your joyful blood begings
To have some relish the yielding. You thank, whatse
For the princes of the maids of our son, you were find
To save which I am day to well the